
# AquaCrisis baselines: stratified 5-fold cross-validation

This notebook evaluates the classical baselines with **out-of-fold (OOF) predictions for every gold-standard post**.

Why this is preferable to one 80/20 split:

- Each supervised model is trained on four folds and predicts the held-out fifth fold.
- Every one of the approximately 1,500 gold posts receives exactly one prediction from a model that did not train on that post.
- The **pooled OOF macro-F1** can therefore be compared directly with LLM macro-F1 measured on the complete gold set.
- The notebook also reports the **mean and standard deviation across the five folds**.

Included baselines:

1. Majority class
2. Fixed keyword rules
3. TF-IDF + class-balanced logistic regression
4. Multilingual sentence embeddings + class-balanced logistic regression

Task B.1 is evaluated by mapping fine-grained Task B gold labels and predictions into the existing broader groups **after prediction**. Thus, the Task B.1 scores remain comparable with the existing post-hoc grouped evaluation.


In [ ]:

# Run this only if the packages are not already installed.
# In Jupyter/Colab, remove the leading "#" from the next line.
# %pip install -q pandas numpy scikit-learn sentence-transformers joblib matplotlib



## 1. Configuration

Set `DATA_PATH_OVERRIDE` and `LABEL_PATH_OVERRIDE` when automatic discovery does not find the CSV files.

Expected text columns:

- `post_id`
- `native_clean`
- `translated_clean`

Expected gold-label columns, either in the main CSV or in a separate label CSV:

- `task_a_gold` or `task_a`
- `task_b_gold` or `task_b`


In [ ]:

from __future__ import annotations

import hashlib
from pathlib import Path
from typing import Iterable

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

try:
    from sentence_transformers import SentenceTransformer
    SENTENCE_TRANSFORMERS_AVAILABLE = True
except ImportError:
    SentenceTransformer = None
    SENTENCE_TRANSFORMERS_AVAILABLE = False


RANDOM_STATE = 42
N_SPLITS = 5

TEXT_VARIANTS = {
    "native": "native_clean",
    "translated": "translated_clean",
}

KEYWORD_VARIANT = "translated"
KEYWORD_TEXT_COLUMN = "translated_clean"

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
EMBEDDING_BATCH_SIZE = 64
FORCE_RECOMPUTE_EMBEDDINGS = False


# Example:
# DATA_PATH_OVERRIDE = Path("post_table_translation_preprocessed.csv")
# LABEL_PATH_OVERRIDE = Path("gold_labels.csv")
DATA_PATH_OVERRIDE = None
LABEL_PATH_OVERRIDE = None

DATA_CANDIDATES = [
    "post_table_translation_preprocessed(3).csv",
    "post_table_translation_preprocessed.csv",
]

LABEL_CANDIDATES = [
    "project-1-at-2026-07-05-10-10-0ebaec54(2).csv",
    "project-1-at-2026-07-05-10-10-0ebaec54.csv",
    "gold_labels.csv",
]

SEARCH_ROOTS = [
    Path.cwd(),
    Path("/workspace"),
    Path("/workspace/.cache"),
    Path("/content"),
    Path("/mnt/data"),
]

OUTPUT_DIR = Path("baseline_outputs_5fold")
EMBEDDING_CACHE_DIR = OUTPUT_DIR / "embedding_cache"
MODEL_DIR = OUTPUT_DIR / "fold_models"
EVAL_DIR = OUTPUT_DIR / "evaluation"

for directory in [OUTPUT_DIR, EMBEDDING_CACHE_DIR, MODEL_DIR, EVAL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR.resolve())


## 2. Labels, Task B.1 mapping, and fixed keyword rules

In [ ]:

TASK_A_LABELS = [
    "Water quality / safety / public health",
    "Service disruption / repair / infrastructure",
    "Water conservation / drought / demand management",
    "Flood / stormwater / wastewater / sewer",
    "Environmental sustainability / waste / recycling / biodiversity",
    "Public education / heritage / community engagement",
    "Routine / institutional / customer service / other",
]

TASK_B_LABELS = [
    "Alert / warning",
    "Instruction / advice to public",
    "Operational update / resolution",
    "Reassurance / safety information",
    "Education / awareness",
    "Institutional promotion / community news",
    "Other / unclear",
]

TASK_B_GROUP_LABELS = [
    "Risk / incident communication",
    "Education / awareness",
    "Institutional / community communication",
    "Other / unclear",
]

TASK_B_GROUP_MAP = {
    "Alert / warning": "Risk / incident communication",
    "Instruction / advice to public": "Risk / incident communication",
    "Operational update / resolution": "Risk / incident communication",
    "Reassurance / safety information": "Risk / incident communication",
    "Education / awareness": "Education / awareness",
    "Institutional promotion / community news":
        "Institutional / community communication",
    "Other / unclear": "Other / unclear",
}

TASK_LABELS = {
    "task_a": TASK_A_LABELS,
    "task_b": TASK_B_LABELS,
    "task_b_grouped": TASK_B_GROUP_LABELS,
}

TASK_A_KEYWORD_RULES = {
    "Water quality / safety / public health": [
        "water quality", "drinking water", "tap water", "safe to drink",
        "water safety", "contamination", "contaminated", "pollution",
        "turbidity", "chlorine", "bacteria", "e coli", "ecoli",
        "coliform", "legionella", "pfas", "boil water", "boil",
        "water sampling", "water monitoring", "quality monitoring",
        "public health", "health risk", "water test", "water testing",
    ],
    "Service disruption / repair / infrastructure": [
        "water interruption", "service interruption", "interruption",
        "disruption", "outage", "no water", "water cut", "low pressure",
        "pipe burst", "burst pipe", "broken pipe", "leak", "fault",
        "repair", "maintenance", "construction", "water network",
        "restoration time", "planned maintenance", "emergency repair",
    ],
    "Water conservation / drought / demand management": [
        "drought", "scarcity", "water scarcity", "water restriction",
        "water restrictions", "water conservation", "water saving",
        "save water", "conserve water", "water consumption",
        "water demand", "heatwave", "heat wave", "dry spell",
        "watering ban",
    ],
    "Flood / stormwater / wastewater / sewer": [
        "flood", "flooding", "stormwater", "storm water", "heavy rain",
        "rainfall", "runoff", "drainage", "sewer", "sewerage",
        "wastewater", "wastewater treatment", "overflow", "sewage",
        "treatment plant",
    ],
    "Environmental sustainability / waste / recycling / biodiversity": [
        "waste sorting", "waste management", "waste collection",
        "recycling", "hazardous waste", "plastic waste", "food waste",
        "biowaste", "battery", "circular economy", "biodiversity",
        "wildlife", "ecosystem", "air quality",
        "environmental conservation", "sustainability", "climate change",
        "invasive species", "compost",
    ],
    "Public education / heritage / community engagement": [
        "education", "educational", "awareness", "campaign", "outreach",
        "school", "children", "student", "workshop", "training", "tour",
        "museum", "exhibition", "world water day", "public awareness",
        "heritage", "history",
    ],
    "Routine / institutional / customer service / other": [
        "holiday", "christmas", "greeting", "celebration", "anniversary",
        "award", "event", "partnership", "customer service", "billing",
        "bill", "payment", "job", "recruitment", "career", "contest",
        "opening hours", "office closed",
    ],
}

TASK_B_KEYWORD_RULES = {
    "Alert / warning": [
        "warning", "alert", "urgent", "emergency", "do not drink",
        "do not use", "risk", "danger", "unsafe", "interruption",
        "outage", "flood warning", "service alert",
    ],
    "Instruction / advice to public": [
        "please", "boil", "boil water", "save water", "conserve water",
        "avoid", "do not", "report", "contact", "prepare", "store water",
        "follow", "use less", "sort", "recycle", "bring", "check",
    ],
    "Operational update / resolution": [
        "update", "updated", "ongoing", "resolved", "fixed", "restored",
        "completed", "repair work", "restoration", "back to normal",
        "service has returned", "work continues",
    ],
    "Reassurance / safety information": [
        "safe", "safe to drink", "no risk", "no contamination",
        "monitoring", "tested", "test results", "under control",
        "normal", "quality is good",
    ],
    "Education / awareness": [
        "learn", "education", "educational", "awareness", "did you know",
        "tips", "guide", "explains", "how", "why", "campaign",
        "workshop", "school", "students",
    ],
    "Institutional promotion / community news": [
        "event", "join us", "celebration", "holiday", "award", "proud",
        "partnership", "recruitment", "job", "career", "museum",
        "exhibition", "festival", "community",
    ],
}

TASK_CONFIG = {
    "task_a": {
        "label_col": "task_a_gold",
        "labels": TASK_A_LABELS,
        "rules": TASK_A_KEYWORD_RULES,
        "keyword_default":
            "Routine / institutional / customer service / other",
    },
    "task_b": {
        "label_col": "task_b_gold",
        "labels": TASK_B_LABELS,
        "rules": TASK_B_KEYWORD_RULES,
        "keyword_default": "Other / unclear",
    },
}


## 3. Load and validate the gold-standard data

In [ ]:

def resolve_existing_path(
    override: Path | str | None,
    candidates: Iterable[str],
    required: bool = True,
) -> Path | None:
    # Resolve a user override first, then search common notebook locations.
    tried = []

    if override is not None:
        path = Path(override).expanduser()
        tried.append(str(path))
        if path.exists():
            return path
        raise FileNotFoundError(f"Configured path does not exist: {path}")

    for candidate in candidates:
        raw_path = Path(candidate)
        possible_paths = (
            [raw_path]
            if raw_path.is_absolute()
            else [root / raw_path for root in SEARCH_ROOTS]
        )
        for path in possible_paths:
            tried.append(str(path))
            if path.exists():
                return path

    if required:
        raise FileNotFoundError(
            "Could not find a required file. Set the corresponding "
            "*_PATH_OVERRIDE variable.\nTried:\n- "
            + "\n- ".join(dict.fromkeys(tried))
        )
    return None


def clean_label_series(series: pd.Series) -> pd.Series:
    return series.fillna("").astype(str).str.strip()


def first_nonempty_label(
    frame: pd.DataFrame,
    candidates: Iterable[str],
) -> pd.Series:
    output = pd.Series("", index=frame.index, dtype=object)
    for column in candidates:
        if column in frame.columns:
            values = clean_label_series(frame[column])
            mask = output.eq("") & values.ne("")
            output.loc[mask] = values.loc[mask]
    return output


def map_task_b_groups(values: Iterable[str]) -> np.ndarray:
    series = pd.Series(list(values), dtype=object)
    series = series.fillna("").astype(str).str.strip()
    mapped = series.map(TASK_B_GROUP_MAP)
    if mapped.isna().any():
        unknown = sorted(series[mapped.isna()].unique())
        raise ValueError(f"Unmapped Task B labels: {unknown}")
    return mapped.to_numpy(dtype=object)


def load_gold_data() -> tuple[pd.DataFrame, Path, Path | None]:
    data_path = resolve_existing_path(
        DATA_PATH_OVERRIDE,
        DATA_CANDIDATES,
        required=True,
    )
    label_path = resolve_existing_path(
        LABEL_PATH_OVERRIDE,
        LABEL_CANDIDATES,
        required=False,
    )

    data = pd.read_csv(data_path, dtype={"post_id": str})
    if "post_id" not in data.columns:
        raise ValueError("The main data CSV must contain a post_id column.")

    data["post_id"] = data["post_id"].astype(str).str.strip()
    data = data.drop_duplicates("post_id").reset_index(drop=True)

    for text_col in set(TEXT_VARIANTS.values()) | {KEYWORD_TEXT_COLUMN}:
        if text_col not in data.columns:
            raise ValueError(
                f"Missing text column {text_col!r} in {data_path}."
            )
        data[text_col] = data[text_col].fillna("").astype(str)

    if label_path is not None:
        labels = pd.read_csv(label_path, dtype={"post_id": str})
        if "post_id" not in labels.columns:
            raise ValueError("The label CSV must contain a post_id column.")

        labels["post_id"] = labels["post_id"].astype(str).str.strip()
        labels["task_a_gold"] = first_nonempty_label(
            labels, ["task_a_gold", "task_a"]
        )
        labels["task_b_gold"] = first_nonempty_label(
            labels, ["task_b_gold", "task_b"]
        )
        labels = labels[
            ["post_id", "task_a_gold", "task_b_gold"]
        ].drop_duplicates("post_id")

        data = data.drop(
            columns=["task_a_gold", "task_b_gold", "task_a", "task_b"],
            errors="ignore",
        )
        data = data.merge(labels, on="post_id", how="left")
    else:
        data["task_a_gold"] = first_nonempty_label(
            data, ["task_a_gold", "task_a"]
        )
        data["task_b_gold"] = first_nonempty_label(
            data, ["task_b_gold", "task_b"]
        )

    data["task_a_gold"] = clean_label_series(data["task_a_gold"])
    data["task_b_gold"] = clean_label_series(data["task_b_gold"])
    data["source_index"] = np.arange(len(data), dtype=int)

    for task_name, config in TASK_CONFIG.items():
        label_col = config["label_col"]
        valid_labels = set(config["labels"])
        nonempty = data[label_col].ne("")
        invalid = sorted(
            data.loc[
                nonempty & ~data[label_col].isin(valid_labels),
                label_col,
            ].unique()
        )
        if invalid:
            print(
                f"Warning: {task_name} has labels outside the configured "
                f"schema; they will be excluded: {invalid}"
            )

    return data, data_path, label_path


df, DATA_PATH, LABEL_PATH = load_gold_data()

print("Data path:", DATA_PATH)
print("Label path:", LABEL_PATH)
print("Rows in main dataset:", len(df))

for task_name, config in TASK_CONFIG.items():
    label_col = config["label_col"]
    labels = config["labels"]
    task_rows = df[df[label_col].isin(labels)]
    print(f"\n{task_name}: {len(task_rows)} valid gold rows")
    display(
        task_rows[label_col]
        .value_counts()
        .reindex(labels, fill_value=0)
        .rename_axis("label")
        .reset_index(name="count")
    )

if not SENTENCE_TRANSFORMERS_AVAILABLE:
    print(
        "\nNote: sentence-transformers is not installed. "
        "The TF-IDF, majority, and keyword baselines will still run. "
        "Install sentence-transformers before running the embedding section."
    )



## 4. Evaluation utilities

The notebook stores:

- one row per fold in `fold_metrics.csv`;
- one OOF prediction per gold item and model in `oof_predictions.csv`;
- pooled OOF metrics and fold mean ± standard deviation in `cv_summary.csv`;
- pooled per-class results in `pooled_per_class.csv`.

Macro-F1 is calculated with the complete configured label set and `zero_division=0`.


In [ ]:

fold_metric_rows: list[dict] = []
oof_prediction_rows: list[dict] = []
fold_assignment_rows: list[dict] = []


def make_task_frame(task_name: str) -> pd.DataFrame:
    config = TASK_CONFIG[task_name]
    label_col = config["label_col"]
    labels = config["labels"]

    task_frame = df[df[label_col].isin(labels)].copy()
    task_frame = task_frame.reset_index(drop=True)

    class_counts = task_frame[label_col].value_counts()
    smallest_class = int(class_counts.min())
    if smallest_class < N_SPLITS:
        raise ValueError(
            f"{task_name}: the smallest class has only {smallest_class} "
            f"examples, fewer than N_SPLITS={N_SPLITS}. "
            "Reduce N_SPLITS or merge sparse classes."
        )
    return task_frame


def make_stratified_splits(
    task_frame: pd.DataFrame,
    label_col: str,
) -> list[tuple[np.ndarray, np.ndarray]]:
    splitter = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    y = task_frame[label_col].to_numpy(dtype=object)
    return list(splitter.split(np.zeros(len(task_frame)), y))


def compute_metrics(
    y_true: Iterable[str],
    y_pred: Iterable[str],
    labels: list[str],
) -> dict:
    y_true = np.asarray(list(y_true), dtype=object)
    y_pred = np.asarray(list(y_pred), dtype=object)

    return {
        "n_eval": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=labels,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=labels,
                average="weighted",
                zero_division=0,
            )
        ),
    }


def record_one_task_fold(
    *,
    task_name: str,
    model_name: str,
    variant: str,
    fold_number: int,
    test_frame: pd.DataFrame,
    y_true: Iterable[str],
    y_pred: Iterable[str],
    labels: list[str],
    trained_on_task: str,
) -> None:
    y_true = np.asarray(list(y_true), dtype=object)
    y_pred = np.asarray(list(y_pred), dtype=object)

    metrics = compute_metrics(y_true, y_pred, labels)
    fold_metric_rows.append(
        {
            "task": task_name,
            "trained_on_task": trained_on_task,
            "model": model_name,
            "variant": variant,
            "fold": int(fold_number),
            **metrics,
        }
    )

    for row, gold, prediction in zip(
        test_frame.itertuples(index=False),
        y_true,
        y_pred,
    ):
        oof_prediction_rows.append(
            {
                "source_index": int(row.source_index),
                "post_id": str(row.post_id),
                "task": task_name,
                "trained_on_task": trained_on_task,
                "model": model_name,
                "variant": variant,
                "fold": int(fold_number),
                "gold": gold,
                "prediction": prediction,
                "correct": bool(gold == prediction),
            }
        )


def record_fold(
    *,
    task_name: str,
    model_name: str,
    variant: str,
    fold_number: int,
    test_frame: pd.DataFrame,
    y_true: Iterable[str],
    y_pred: Iterable[str],
) -> None:
    labels = TASK_LABELS[task_name]

    record_one_task_fold(
        task_name=task_name,
        model_name=model_name,
        variant=variant,
        fold_number=fold_number,
        test_frame=test_frame,
        y_true=y_true,
        y_pred=y_pred,
        labels=labels,
        trained_on_task=task_name,
    )

    if task_name == "task_b":
        grouped_true = map_task_b_groups(y_true)
        grouped_pred = map_task_b_groups(y_pred)

        record_one_task_fold(
            task_name="task_b_grouped",
            model_name=model_name,
            variant=variant,
            fold_number=fold_number,
            test_frame=test_frame,
            y_true=grouped_true,
            y_pred=grouped_pred,
            labels=TASK_B_GROUP_LABELS,
            trained_on_task="task_b",
        )


def keyword_predict(
    text: str,
    rules: dict[str, list[str]],
    default_label: str,
) -> str:
    normalized = str(text).lower()
    for label, keywords in rules.items():
        if any(keyword.lower() in normalized for keyword in keywords):
            return label
    return default_label


def build_tfidf_logreg() -> Pipeline:
    return Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    max_features=30000,
                    ngram_range=(1, 2),
                    min_df=2,
                ),
            ),
            (
                "lr",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def build_embedding_logreg() -> LogisticRegression:
    return LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )


task_frames = {
    task_name: make_task_frame(task_name)
    for task_name in TASK_CONFIG
}

task_splits = {}
for task_name, task_frame in task_frames.items():
    label_col = TASK_CONFIG[task_name]["label_col"]
    task_splits[task_name] = make_stratified_splits(
        task_frame,
        label_col,
    )

    fold_for_row = np.full(len(task_frame), -1, dtype=int)
    for fold_number, (_, test_idx) in enumerate(
        task_splits[task_name],
        start=1,
    ):
        fold_for_row[test_idx] = fold_number

    for row, fold_number in zip(
        task_frame.itertuples(index=False),
        fold_for_row,
    ):
        fold_assignment_rows.append(
            {
                "task": task_name,
                "source_index": int(row.source_index),
                "post_id": str(row.post_id),
                "fold": int(fold_number),
            }
        )

print("Created reproducible stratified folds.")


## 5. Majority and keyword baselines

In [ ]:

for task_name, config in TASK_CONFIG.items():
    task_frame = task_frames[task_name]
    label_col = config["label_col"]
    y = task_frame[label_col].to_numpy(dtype=object)

    for fold_number, (train_idx, test_idx) in enumerate(
        task_splits[task_name],
        start=1,
    ):
        train_frame = task_frame.iloc[train_idx]
        test_frame = task_frame.iloc[test_idx]

        majority_label = (
            train_frame[label_col].value_counts().idxmax()
        )
        majority_pred = np.repeat(majority_label, len(test_idx))

        record_fold(
            task_name=task_name,
            model_name="majority",
            variant="all",
            fold_number=fold_number,
            test_frame=test_frame,
            y_true=y[test_idx],
            y_pred=majority_pred,
        )

        keyword_pred = (
            test_frame[KEYWORD_TEXT_COLUMN]
            .fillna("")
            .astype(str)
            .apply(
                lambda text: keyword_predict(
                    text,
                    config["rules"],
                    config["keyword_default"],
                )
            )
            .to_numpy(dtype=object)
        )

        record_fold(
            task_name=task_name,
            model_name="keyword_rules",
            variant=KEYWORD_VARIANT,
            fold_number=fold_number,
            test_frame=test_frame,
            y_true=y[test_idx],
            y_pred=keyword_pred,
        )

print("Finished majority and keyword baselines.")


## 6. TF-IDF + logistic regression with fold-local feature fitting

In [ ]:

for task_name, config in TASK_CONFIG.items():
    task_frame = task_frames[task_name]
    label_col = config["label_col"]
    y = task_frame[label_col].to_numpy(dtype=object)

    for variant_name, text_col in TEXT_VARIANTS.items():
        print(f"\nTF-IDF + LR | {task_name} | {variant_name}")

        for fold_number, (train_idx, test_idx) in enumerate(
            task_splits[task_name],
            start=1,
        ):
            train_frame = task_frame.iloc[train_idx]
            test_frame = task_frame.iloc[test_idx]

            model = build_tfidf_logreg()
            model.fit(
                train_frame[text_col].fillna("").astype(str),
                y[train_idx],
            )
            predictions = model.predict(
                test_frame[text_col].fillna("").astype(str)
            )

            record_fold(
                task_name=task_name,
                model_name="tfidf_logreg",
                variant=variant_name,
                fold_number=fold_number,
                test_frame=test_frame,
                y_true=y[test_idx],
                y_pred=predictions,
            )

            model_path = (
                MODEL_DIR
                / f"{task_name}_tfidf_logreg_{variant_name}"
                  f"_fold{fold_number}.joblib"
            )
            joblib.dump(model, model_path)

            fold_score = f1_score(
                y[test_idx],
                predictions,
                labels=config["labels"],
                average="macro",
                zero_division=0,
            )
            print(
                f"  fold {fold_number}: "
                f"n={len(test_idx)}, macro-F1={fold_score:.4f}"
            )

print("\nFinished TF-IDF cross-validation.")



## 7. Multilingual sentence embeddings + logistic regression

The pretrained encoder is frozen and does not use the task labels. Embeddings are cached once for the union of gold rows, while the logistic-regression classifier is trained separately within each fold.

The first run may download the sentence-transformer model.


In [ ]:

def text_cache_fingerprint(
    frame: pd.DataFrame,
    text_col: str,
) -> str:
    hasher = hashlib.sha256()
    for post_id, text in zip(frame["post_id"], frame[text_col]):
        hasher.update(str(post_id).encode("utf-8"))
        hasher.update(b"\0")
        hasher.update(str(text).encode("utf-8"))
        hasher.update(b"\n")
    return hasher.hexdigest()[:16]


def get_or_create_gold_embeddings(
    gold_union: pd.DataFrame,
    variant_name: str,
    text_col: str,
    encoder: SentenceTransformer,
) -> np.ndarray:
    fingerprint = text_cache_fingerprint(gold_union, text_col)
    safe_model_name = EMBEDDING_MODEL_NAME.replace("/", "__")
    cache_path = (
        EMBEDDING_CACHE_DIR
        / f"{safe_model_name}_{variant_name}_{fingerprint}.npy"
    )

    if cache_path.exists() and not FORCE_RECOMPUTE_EMBEDDINGS:
        print("Loading cached embeddings:", cache_path)
        embeddings = np.load(cache_path)
        if len(embeddings) != len(gold_union):
            raise ValueError(
                "Cached embedding row count does not match the gold data."
            )
        return embeddings

    print(
        f"Encoding {len(gold_union)} gold rows for {variant_name!r} "
        f"with {EMBEDDING_MODEL_NAME}..."
    )
    embeddings = encoder.encode(
        gold_union[text_col].fillna("").astype(str).tolist(),
        batch_size=EMBEDDING_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    np.save(cache_path, embeddings)
    print("Saved embedding cache:", cache_path)
    return embeddings


if not SENTENCE_TRANSFORMERS_AVAILABLE:
    print(
        "Skipping the embedding baseline because sentence-transformers "
        "is not installed. Install it, restart the kernel if necessary, "
        "and rerun this section."
    )
else:
    gold_mask = (
        df["task_a_gold"].isin(TASK_A_LABELS)
        | df["task_b_gold"].isin(TASK_B_LABELS)
    )
    gold_union = (
        df.loc[gold_mask]
        .copy()
        .sort_values("source_index")
        .reset_index(drop=True)
    )
    source_to_embedding_row = {
        int(source_index): position
        for position, source_index
        in enumerate(gold_union["source_index"].tolist())
    }

    encoder = SentenceTransformer(EMBEDDING_MODEL_NAME)

    embeddings_by_variant = {}
    for variant_name, text_col in TEXT_VARIANTS.items():
        embeddings_by_variant[variant_name] = (
            get_or_create_gold_embeddings(
                gold_union,
                variant_name,
                text_col,
                encoder,
            )
        )

    for task_name, config in TASK_CONFIG.items():
        task_frame = task_frames[task_name]
        label_col = config["label_col"]
        y = task_frame[label_col].to_numpy(dtype=object)

        embedding_rows = np.array(
            [
                source_to_embedding_row[int(source_index)]
                for source_index in task_frame["source_index"]
            ],
            dtype=int,
        )

        for variant_name in TEXT_VARIANTS:
            x = embeddings_by_variant[variant_name][embedding_rows]
            print(
                f"\nMultilingual embeddings + LR | "
                f"{task_name} | {variant_name}"
            )

            for fold_number, (train_idx, test_idx) in enumerate(
                task_splits[task_name],
                start=1,
            ):
                test_frame = task_frame.iloc[test_idx]

                classifier = build_embedding_logreg()
                classifier.fit(x[train_idx], y[train_idx])
                predictions = classifier.predict(x[test_idx])

                record_fold(
                    task_name=task_name,
                    model_name="multilingual_embeddings",
                    variant=variant_name,
                    fold_number=fold_number,
                    test_frame=test_frame,
                    y_true=y[test_idx],
                    y_pred=predictions,
                )

                model_path = (
                    MODEL_DIR
                    / f"{task_name}_multilingual_embeddings_{variant_name}"
                      f"_fold{fold_number}.joblib"
                )
                joblib.dump(classifier, model_path)

                fold_score = f1_score(
                    y[test_idx],
                    predictions,
                    labels=config["labels"],
                    average="macro",
                    zero_division=0,
                )
                print(
                    f"  fold {fold_number}: "
                    f"n={len(test_idx)}, macro-F1={fold_score:.4f}"
                )

    print("\nFinished multilingual-embedding cross-validation.")


## 8. Aggregate pooled OOF scores, fold variability, and per-class metrics

In [ ]:

def validate_oof_predictions(oof_df: pd.DataFrame) -> None:
    if oof_df.empty:
        raise RuntimeError("No OOF predictions were generated.")

    key_cols = ["task", "model", "variant", "source_index"]
    duplicates = oof_df.duplicated(key_cols, keep=False)
    if duplicates.any():
        example = oof_df.loc[duplicates, key_cols].head()
        raise ValueError(
            "Some task/model/variant combinations predicted a gold row "
            f"more than once:\n{example}"
        )


def pooled_per_class_metrics(
    oof_df: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for (task, model, variant), group in oof_df.groupby(
        ["task", "model", "variant"],
        sort=False,
    ):
        labels = TASK_LABELS[task]
        precision, recall, f1, support = precision_recall_fscore_support(
            group["gold"],
            group["prediction"],
            labels=labels,
            zero_division=0,
        )

        for label, p, r, score, n in zip(
            labels,
            precision,
            recall,
            f1,
            support,
        ):
            rows.append(
                {
                    "task": task,
                    "model": model,
                    "variant": variant,
                    "label": label,
                    "precision": float(p),
                    "recall": float(r),
                    "f1": float(score),
                    "support": int(n),
                }
            )

    return pd.DataFrame(rows)


def build_cv_summary(
    fold_df: pd.DataFrame,
    oof_df: pd.DataFrame,
) -> pd.DataFrame:
    fold_summary = (
        fold_df
        .groupby(["task", "model", "variant"], as_index=False)
        .agg(
            folds=("fold", "nunique"),
            fold_n_eval_mean=("n_eval", "mean"),
            fold_accuracy_mean=("accuracy", "mean"),
            fold_accuracy_std=("accuracy", "std"),
            fold_macro_f1_mean=("macro_f1", "mean"),
            fold_macro_f1_std=("macro_f1", "std"),
            fold_weighted_f1_mean=("weighted_f1", "mean"),
            fold_weighted_f1_std=("weighted_f1", "std"),
        )
    )

    pooled_rows = []
    for (task, model, variant), group in oof_df.groupby(
        ["task", "model", "variant"],
        sort=False,
    ):
        metrics = compute_metrics(
            group["gold"],
            group["prediction"],
            TASK_LABELS[task],
        )
        pooled_rows.append(
            {
                "task": task,
                "model": model,
                "variant": variant,
                "pooled_n_eval": metrics["n_eval"],
                "pooled_accuracy": metrics["accuracy"],
                "pooled_macro_f1": metrics["macro_f1"],
                "pooled_weighted_f1": metrics["weighted_f1"],
            }
        )

    pooled_df = pd.DataFrame(pooled_rows)
    return (
        fold_summary
        .merge(
            pooled_df,
            on=["task", "model", "variant"],
            how="inner",
        )
        .sort_values(
            ["task", "pooled_macro_f1"],
            ascending=[True, False],
        )
        .reset_index(drop=True)
    )


fold_metrics = pd.DataFrame(fold_metric_rows)
oof_predictions = pd.DataFrame(oof_prediction_rows)
fold_assignments = pd.DataFrame(fold_assignment_rows)

validate_oof_predictions(oof_predictions)

cv_summary = build_cv_summary(fold_metrics, oof_predictions)
per_class = pooled_per_class_metrics(oof_predictions)

coverage_check = (
    oof_predictions
    .groupby(["task", "model", "variant"], as_index=False)
    .agg(
        oof_rows=("source_index", "size"),
        unique_gold_rows=("source_index", "nunique"),
        folds=("fold", "nunique"),
    )
)

expected_rows = {
    "task_a": len(task_frames["task_a"]),
    "task_b": len(task_frames["task_b"]),
    "task_b_grouped": len(task_frames["task_b"]),
}
coverage_check["expected_gold_rows"] = (
    coverage_check["task"].map(expected_rows)
)
coverage_check["complete_oof_coverage"] = (
    coverage_check["unique_gold_rows"]
    == coverage_check["expected_gold_rows"]
)

if not coverage_check["complete_oof_coverage"].all():
    display(
        coverage_check.loc[
            ~coverage_check["complete_oof_coverage"]
        ]
    )
    raise ValueError("Incomplete OOF prediction coverage detected.")

fold_metrics.to_csv(EVAL_DIR / "fold_metrics.csv", index=False)
oof_predictions.to_csv(
    EVAL_DIR / "oof_predictions.csv",
    index=False,
)
fold_assignments.to_csv(
    EVAL_DIR / "fold_assignments.csv",
    index=False,
)
cv_summary.to_csv(EVAL_DIR / "cv_summary.csv", index=False)
per_class.to_csv(
    EVAL_DIR / "pooled_per_class.csv",
    index=False,
)
coverage_check.to_csv(
    EVAL_DIR / "oof_coverage_check.csv",
    index=False,
)

print("Saved evaluation files to:", EVAL_DIR.resolve())
display(coverage_check)
display(cv_summary.round(4))



## 9. Paper-ready comparison table

For direct comparison with LLM results evaluated on all gold posts, use **`pooled_macro_f1`**. Report the fold mean and standard deviation as an additional stability estimate.

Avoid using scores from a model fitted and evaluated on all 1,500 rows.


In [ ]:

paper_table = cv_summary[
    [
        "task",
        "model",
        "variant",
        "pooled_macro_f1",
        "fold_macro_f1_mean",
        "fold_macro_f1_std",
        "pooled_accuracy",
        "pooled_weighted_f1",
        "pooled_n_eval",
    ]
].copy()

paper_table["macro_f1_mean_sd"] = paper_table.apply(
    lambda row: (
        f"{row['fold_macro_f1_mean']:.3f} "
        f"± {row['fold_macro_f1_std']:.3f}"
    ),
    axis=1,
)

paper_table["pooled_macro_f1"] = (
    paper_table["pooled_macro_f1"].round(3)
)
paper_table["pooled_accuracy"] = (
    paper_table["pooled_accuracy"].round(3)
)
paper_table["pooled_weighted_f1"] = (
    paper_table["pooled_weighted_f1"].round(3)
)

paper_table = paper_table[
    [
        "task",
        "model",
        "variant",
        "pooled_macro_f1",
        "macro_f1_mean_sd",
        "pooled_accuracy",
        "pooled_weighted_f1",
        "pooled_n_eval",
    ]
]

paper_table.to_csv(
    EVAL_DIR / "paper_comparison_macro_f1.csv",
    index=False,
)

display(paper_table)



## 10. Optional: append the LLM results from your 1,500-post evaluation

Edit `LLM_RESULTS` with the final scores from the same gold-label version. The resulting table places LLM macro-F1 beside the baseline pooled OOF macro-F1.

The baseline `pooled_macro_f1` and the LLM `macro_f1` are both calculated over the complete gold set, but they represent different learning settings:

- supervised baselines: 5-fold OOF predictions;
- LLMs: zero-shot or five-shot inference.


In [ ]:

LLM_RESULTS = [
    # Example:
    # {
    #     "task": "task_a",
    #     "model": "GPT-4o-mini",
    #     "variant": "native_zero_shot",
    #     "macro_f1": 0.000,
    # },
]

if LLM_RESULTS:
    llm_df = pd.DataFrame(LLM_RESULTS)
    llm_df["evaluation"] = "LLM on complete gold set"
    llm_df = llm_df.rename(
        columns={"macro_f1": "comparable_macro_f1"}
    )

    baseline_df = cv_summary[
        ["task", "model", "variant", "pooled_macro_f1"]
    ].copy()
    baseline_df["evaluation"] = "5-fold pooled OOF"
    baseline_df = baseline_df.rename(
        columns={"pooled_macro_f1": "comparable_macro_f1"}
    )

    combined_comparison = pd.concat(
        [baseline_df, llm_df],
        ignore_index=True,
        sort=False,
    ).sort_values(
        ["task", "comparable_macro_f1"],
        ascending=[True, False],
    )

    combined_comparison.to_csv(
        EVAL_DIR / "baseline_llm_comparison.csv",
        index=False,
    )
    display(combined_comparison.round(4))
else:
    print(
        "LLM_RESULTS is empty. Add the final LLM scores above to create "
        "baseline_llm_comparison.csv."
    )



## Recommended reporting language

> We evaluated the supervised baselines using stratified five-fold cross-validation on the 1,500-post gold standard. In each fold, models were trained on 80% of the annotated posts and evaluated on the held-out 20%, with label proportions preserved. We pooled the out-of-fold predictions so that every gold-standard post was evaluated exactly once by a model that had not been trained on that post. We report pooled macro-F1 for comparison with the LLMs evaluated on the complete gold set, together with the mean and standard deviation of macro-F1 across folds.

For Task B.1, add:

> Task B.1 scores were computed post hoc by mapping fine-grained Task B gold labels and out-of-fold predictions to the broader communication-function groups.
